#Cargar el modelo y el preprocesador

In [1]:
import pandas as pd
import numpy as np
import joblib

# Cargar modelo entrenado
modelo = joblib.load("../models/modelo_logistico.joblib")

# Cargar preprocesador
preprocesador = joblib.load("../models/preprocesador.joblib")

print("Modelo cargado correctamente.")
print("Preprocesador cargado correctamente.")

Modelo cargado correctamente.
Preprocesador cargado correctamente.


In [2]:
import pandas as pd

df = pd.read_csv("../data/partidos_2024_limpio.csv")

# Mantener partidos completados y excluir empates
df = df[
    df["homePoints"].notna() &
    df["awayPoints"].notna() &
    (df["homePoints"] != df["awayPoints"])
].copy()

# Crear variables de fecha
df["startDate"] = pd.to_datetime(df["startDate"])
df["month"] = df["startDate"].dt.month
df["dayOfWeek"] = df["startDate"].dt.dayofweek

print("Datos preparados:", df.shape)

Datos preparados: (3786, 36)


C:\Users\maxwe\AppData\Local\Temp\ipykernel_41188\1310312538.py:15: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["dayOfWeek"] = df["startDate"].dt.dayofweek


In [3]:
features = [
    "week", "seasonType", "neutralSite", "conferenceGame",
    "homeId", "awayId", "homeClassification", "awayClassification",
    "homeConference", "awayConference", "startTimeTBD",
    "month", "dayOfWeek"
]

partido = df[df["week"].between(13, 16)].iloc[[0]]

X_ejemplo = partido[features]

print(X_ejemplo.T)

                            3459
week                          13
seasonType               regular
neutralSite                False
conferenceGame              True
homeId                      2309
awayId                      2006
homeClassification           fbs
awayClassification           fbs
homeConference      Mid-American
awayConference      Mid-American
startTimeTBD               False
month                         11
dayOfWeek                      2


Realizar la predicción

In [4]:
# Transformar los datos con el preprocesador
X_ejemplo_encoded = preprocesador.transform(X_ejemplo)

# Predecir resultado
prediccion = modelo.predict(X_ejemplo_encoded)

# Obtener probabilidades
probabilidades = modelo.predict_proba(X_ejemplo_encoded)[0]

print("Predicción:", "Gana el equipo local" if prediccion[0] == 1 else "Gana el equipo visitante")
print(f"Probabilidad de victoria local: {probabilidades[1]:.2%}")
print(f"Probabilidad de victoria visitante: {probabilidades[0]:.2%}")

Predicción: Gana el equipo visitante
Probabilidad de victoria local: 39.34%
Probabilidad de victoria visitante: 60.66%


comprobar si la predicción acertó.

In [6]:
# Calcular el resultado real utilizando los puntos
resultado_real = int(
    partido["homePoints"].iloc[0] > partido["awayPoints"].iloc[0]
)

# Obtener la predicción del modelo
resultado_predicho = int(prediccion[0])

print("Resultado real:",
      "Victoria local" if resultado_real == 1 else "Victoria visitante")

print("Resultado predicho:",
      "Victoria local" if resultado_predicho == 1 else "Victoria visitante")

print("¿El modelo acertó?:",
      "Sí" if resultado_real == resultado_predicho else "No")

Resultado real: Victoria visitante
Resultado predicho: Victoria visitante
¿El modelo acertó?: Sí


guardar predicción

In [7]:
import os

os.makedirs("../reports", exist_ok=True)

resultado_final = pd.DataFrame([{
    "partido_id": partido.index[0],
    "resultado_real": resultado_real,
    "resultado_predicho": resultado_predicho,
    "probabilidad_local": probabilidades[1],
    "probabilidad_visitante": probabilidades[0],
    "acierto": resultado_real == resultado_predicho
}])

resultado_final.to_csv(
    "../reports/prediccion_prueba.csv",
    index=False
)

print("Predicción guardada correctamente.")

Predicción guardada correctamente.
